In [ ]:
import pkgutil, importlib
import cuda.bindings as _pkg_under_test
# DENYLIST (minimal): cuda.bindings._test_helpers.* hard-require `pytest` — a test-only
# dependency that may not be installed. They are the project's own pytest
# test-scaffolding shipped in the wheel, NOT a cuda_bindings defect. Everything else
# (incl. every GPU-touching submodule) imports GPU-less: the CUDA .so dlopen is LAZY,
# fired on the first API call, so importing the module never needs a driver/GPU.
_DENY = frozenset({"cuda.bindings._test_helpers"})
failed = []
for module_info in pkgutil.walk_packages(_pkg_under_test.__path__, prefix=f"{_pkg_under_test.__name__}."):
    if any(module_info.name == d or module_info.name.startswith(d + ".") for d in _DENY):
        continue
    try:
        importlib.import_module(module_info.name)
    except Exception as e:
        failed.append((module_info.name, type(e).__name__, str(e)))
if failed:
    for name, etype, msg in failed:
        print(f"FAIL {name}: {etype}: {msg}")
    raise AssertionError(f"{len(failed)} submodule(s) of {_pkg_under_test.__name__} failed to import")
print(f"OK — all submodules of {_pkg_under_test.__name__} imported cleanly")

## Example 1 — Version introspection (real runtime init, GPU-less)

In [ ]:
# Example 1 — version introspection across the runtime + NVRTC libs (all CPU-safe).
import importlib.metadata as md

import cuda.bindings
from cuda.bindings import runtime as rt, nvrtc

from packaging.version import Version

bindings_major = Version(cuda.bindings.__version__).major  # cuda-bindings N.x pairs with CUDA N.x
print("cuda.bindings.__version__:", cuda.bindings.__version__)
assert md.version("cuda-bindings") == cuda.bindings.__version__, (
    f'wheel metadata {md.version("cuda-bindings")} disagrees with the module '
    f"{cuda.bindings.__version__}"
)

err, rt_ver = rt.cudaRuntimeGetVersion()          # links libcudart -> real runtime init
assert err == rt.cudaError_t.cudaSuccess, err
print("CUDA runtime version:", rt_ver)             # e.g. 13030 -> 13.3
assert rt_ver // 1000 == bindings_major, (rt_ver, cuda.bindings.__version__)

err, major, minor = nvrtc.nvrtcVersion()           # dlopens libnvrtc -> real runtime init
assert err == nvrtc.nvrtcResult.NVRTC_SUCCESS, err
print("NVRTC version:", (major, minor))
assert major == bindings_major, (major, cuda.bindings.__version__)

print("Example 1 OK (runtime + NVRTC; the driver half is the next cell)")

In [ ]:
from cuda.bindings import driver as drv

# libcuda.so.1 comes with the NVIDIA driver (inside a container, the NVIDIA Container Toolkit mounts it
# from the host); the marker above skips this cell where it is absent, so here the query must succeed.
err, dv = drv.cuDriverGetVersion()
assert err == drv.CUresult.CUDA_SUCCESS, err
print("CUDA driver version:", dv)
print("Example 1 OK")

## Example 2 — NVRTC: compile CUDA C++ → PTX on CPU (heaviest GPU-less runtime path)

In [ ]:
# Example 2 — NVRTC compile a saxpy kernel to PTX (CPU-only; no GPU/driver needed).
from cuda.bindings import nvrtc

def _ck(err):
    assert err == nvrtc.nvrtcResult.NVRTC_SUCCESS, err

src = b"""
extern "C" __global__ void saxpy(float a, float *x, float *y, float *out, int n) {
  int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) out[i] = a * x[i] + y[i];
}
"""
err, prog = nvrtc.nvrtcCreateProgram(src, b"saxpy.cu", 0, [], []); _ck(err)
opts = [b"--gpu-architecture=compute_75"]
err, = nvrtc.nvrtcCompileProgram(prog, len(opts), opts)
if err != nvrtc.nvrtcResult.NVRTC_SUCCESS:                    # surface the compile log
    _, lsz = nvrtc.nvrtcGetProgramLogSize(prog); log = bytearray(lsz)
    nvrtc.nvrtcGetProgramLog(prog, log); raise AssertionError(bytes(log).decode())
err, ptxsz = nvrtc.nvrtcGetPTXSize(prog); _ck(err)
ptx = bytearray(ptxsz); err, = nvrtc.nvrtcGetPTX(prog, ptx); _ck(err)
nvrtc.nvrtcDestroyProgram(prog)
print("PTX bytes:", ptxsz)
assert b".visible .entry saxpy" in bytes(ptx), "kernel entry missing from PTX"
print("Example 2 OK — compiled saxpy to PTX")

## Example 3 — Driver API: init + device enumeration (GPU-conditional)

`cuInit` + `cuDeviceGet*` are the driver-API entry points. This passes **with and without**
a GPU: on a machine with no visible GPU `cuInit` returns `CUDA_ERROR_NO_DEVICE` (or `libcuda`
is absent entirely), and the cell prints `SKIP (no GPU)`; with a GPU it enumerates each device.
Any other driver error fails the cell.

In [ ]:
# Example 3 — driver-API device enumeration (GPU-conditional).
from cuda.bindings import driver as drv

def _gpu_ready():
    # cuda.bindings reports CUDA errors as return codes; it raises (a RuntimeError) only when it
    # cannot load the driver library libcuda.so.1, i.e. there is no NVIDIA driver on this machine.
    try:
        err, = drv.cuInit(0)
    except RuntimeError as e:
        print(f"SKIP (no GPU): driver library unavailable ({type(e).__name__}: {e})"); return False
    if err == drv.CUresult.CUDA_ERROR_NO_DEVICE:
        print(f"SKIP (no GPU): cuInit -> {err}"); return False
    assert err == drv.CUresult.CUDA_SUCCESS, err      # any other driver error fails the cell
    return True

if _gpu_ready():
    err, count = drv.cuDeviceGetCount(); assert err == drv.CUresult.CUDA_SUCCESS, err
    print(f"CUDA devices: {count}")
    A = drv.CUdevice_attribute
    for i in range(count):
        err, dev = drv.cuDeviceGet(i); assert err == drv.CUresult.CUDA_SUCCESS, err
        err, name = drv.cuDeviceGetName(256, dev); assert err == drv.CUresult.CUDA_SUCCESS, err
        err, cc_hi = drv.cuDeviceGetAttribute(A.CU_DEVICE_ATTRIBUTE_COMPUTE_CAPABILITY_MAJOR, dev)
        assert err == drv.CUresult.CUDA_SUCCESS, err
        err, cc_lo = drv.cuDeviceGetAttribute(A.CU_DEVICE_ATTRIBUTE_COMPUTE_CAPABILITY_MINOR, dev)
        assert err == drv.CUresult.CUDA_SUCCESS, err
        clean = name.split(b"\x00")[0].decode(errors="replace")
        print(f"  GPU {i}: {clean} (sm_{cc_hi}{cc_lo})")
    assert count > 0
print("Example 3 OK")

## Example 4 — Runtime API: device query + host↔device memcpy (GPU-conditional)

`cudaGetDeviceCount` is CPU-safe — it cleanly returns `cudaErrorNoDevice` with count 0 on a
machine without a GPU (a valid result, not a crash), so the query path is always exercised. When a
GPU is present the cell additionally round-trips a buffer through `cudaMalloc`/`cudaMemcpy`.

In [ ]:
# Example 4 — runtime-API device query + GPU-conditional device memcpy roundtrip.
import numpy as np
from cuda.bindings import runtime as rt

err, count = rt.cudaGetDeviceCount()
_no_device = (rt.cudaError_t.cudaErrorNoDevice, rt.cudaError_t.cudaErrorInsufficientDriver)
if err in _no_device or (count or 0) == 0:
    assert err in (rt.cudaError_t.cudaSuccess, *_no_device), err
    print(f"SKIP (no GPU): cudaGetDeviceCount -> {err}, count={count}")
else:
    assert err == rt.cudaError_t.cudaSuccess, err
    print(f"CUDA runtime sees {count} device(s)")
    host = np.arange(16, dtype=np.float32); nbytes = host.nbytes
    err, dptr = rt.cudaMalloc(nbytes); assert err == rt.cudaError_t.cudaSuccess, err
    (err,) = rt.cudaMemcpy(dptr, host.ctypes.data, nbytes, rt.cudaMemcpyKind.cudaMemcpyHostToDevice)
    out = np.empty_like(host)
    (err,) = rt.cudaMemcpy(out.ctypes.data, dptr, nbytes, rt.cudaMemcpyKind.cudaMemcpyDeviceToHost)
    (err,) = rt.cudaFree(dptr)
    assert np.array_equal(host, out), "device roundtrip corrupted data"
    print("  host<->device roundtrip verified")
print("Example 4 OK")

## Example 5 — End-to-end kernel launch (GPU-conditional)

The full runtime path: NVRTC-compile a vector-add kernel, load it through the driver API,
launch it, and verify the result numerically (`1 + 2 == 3`). This is skipped without a GPU
(no device, or no driver library); with one it proves the compile → module-load → launch →
copy-back chain end-to-end, and any CUDA error along the way fails the cell.

In [ ]:
# Example 5 — compile + load + launch a vector-add kernel, verify numerically (GPU-conditional).
import numpy as np
from cuda.bindings import driver as drv, nvrtc

def _drv(result):
    # Unpack a driver-API result tuple, asserting CUDA_SUCCESS; return the value(s) after the status.
    err, *values = result
    assert err == drv.CUresult.CUDA_SUCCESS, err
    return values[0] if len(values) == 1 else values

def _rtc(result):
    err, *values = result
    assert err == nvrtc.nvrtcResult.NVRTC_SUCCESS, err
    return values[0] if len(values) == 1 else values

# Skip only when there is no device to run on: the driver library cannot be loaded (cuda.bindings
# raises a RuntimeError) or cuInit reports CUDA_ERROR_NO_DEVICE. Every other failure raises.
try:
    (err,) = drv.cuInit(0)
except RuntimeError as e:
    err = e
if isinstance(err, RuntimeError) or err == drv.CUresult.CUDA_ERROR_NO_DEVICE:
    print(f"SKIP (no GPU): {err!r} — kernel launch not exercised")
else:
    assert err == drv.CUresult.CUDA_SUCCESS, err
    dev = _drv(drv.cuDeviceGet(0))
    # Run in the device's primary context (the one the runtime API in Example 4 also uses) instead of
    # creating a second context on the same device.
    ctx = _drv(drv.cuDevicePrimaryCtxRetain(dev)); _drv(drv.cuCtxSetCurrent(ctx))
    src = b'extern "C" __global__ void add(float *a,float *b,float *o){int i=threadIdx.x; o[i]=a[i]+b[i];}'
    prog = _rtc(nvrtc.nvrtcCreateProgram(src, b"add.cu", 0, [], []))
    (err,) = nvrtc.nvrtcCompileProgram(prog, 0, [])
    if err != nvrtc.nvrtcResult.NVRTC_SUCCESS:                    # surface the compile log
        lsz = _rtc(nvrtc.nvrtcGetProgramLogSize(prog)); log = bytearray(lsz)
        nvrtc.nvrtcGetProgramLog(prog, log); raise AssertionError(bytes(log).decode())
    sz = _rtc(nvrtc.nvrtcGetPTXSize(prog)); ptx = bytearray(sz); _rtc(nvrtc.nvrtcGetPTX(prog, ptx))
    _rtc(nvrtc.nvrtcDestroyProgram(prog))
    mod = _drv(drv.cuModuleLoadData(bytes(ptx))); fn = _drv(drv.cuModuleGetFunction(mod, b"add"))
    n = 256; a = np.ones(n, np.float32); b = np.full(n, 2, np.float32); o = np.empty(n, np.float32); bs = a.nbytes
    dptrs = [_drv(drv.cuMemAlloc(bs)) for _ in range(3)]
    _drv(drv.cuMemcpyHtoD(dptrs[0], a.ctypes.data, bs)); _drv(drv.cuMemcpyHtoD(dptrs[1], b.ctypes.data, bs))
    holders = [np.array([int(p)], np.uint64) for p in dptrs]          # keep pointer-boxes alive
    kparams = np.array([h.ctypes.data for h in holders], np.uint64)
    _drv(drv.cuLaunchKernel(fn, 1, 1, 1, n, 1, 1, 0, 0, kparams.ctypes.data, 0))
    _drv(drv.cuCtxSynchronize()); _drv(drv.cuMemcpyDtoH(o.ctypes.data, dptrs[2], bs))
    for p in dptrs:
        _drv(drv.cuMemFree(p))
    _drv(drv.cuModuleUnload(mod)); _drv(drv.cuDevicePrimaryCtxRelease(dev))
    assert np.allclose(o, 3.0), o[:8]
    print("Example 5 OK — kernel result verified (1+2==3)")